# WikiLoc Export Control Notebook

## Control

In [51]:
# Control Cell
TEST_ACTIVITY_URL_EXPORT = True
TEST_ACTIVITY_EXPORT = False
AUTO_ACTIVITY_EXPORT = False

## Setup

In [52]:
# pip install playwright
# python -m playwright install chromium

import random
import subprocess
import sys
import time
from pathlib import Path
import pandas as pd

# Random pause range (minutes) between automated export calls
MIN_DELAY_MINUTES = 0.2
MAX_DELAY_MINUTES = 0.6

In [53]:
WIKILOC_MAP_URL = "https://www.wikiloc.com/wikiloc/map.do?sw=-43.993308489691415%2C172.5656890869141&ne=-43.56496912804993%2C173.40957641601565&sort=last&page=1"
WIKILOC_ACTIVITY_URL = "https://www.wikiloc.com/wikiloc/download.do?id=191234223"
OUT_DIR = Path("Data\\WikiLoc_Downloads")

# Reuse a persistent browser profile so login/challenge can be solved once and reused
PROFILE_DIR = Path("Data\\wikiloc_profile")
STORAGE_STATE_FILE = Path("Data\\wikiloc_storage_state.json")

# Shared activity index created from map cards
WIKILOC_INDEX_CSV = OUT_DIR / "wikiloc_activity_index.csv"

## Testing Calls

In [54]:
if TEST_ACTIVITY_URL_EXPORT:
    try:
        result = subprocess.run(
            [
                sys.executable,
                "wikiloc_activity_urls_export.py",
                "--url",
                WIKILOC_MAP_URL,
                "--out-dir",
                str(OUT_DIR),
                "--profile-dir",
                str(PROFILE_DIR),
                "--storage-state",
                str(STORAGE_STATE_FILE),
                "--index-csv",
                str(WIKILOC_INDEX_CSV),
                "--start-page",
                "30",
                "--max-page",
                "31",
                "--slow-mo-ms",
                "0",
            ],
            check=True,
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr)
        raise

Loading page 30: https://www.wikiloc.com/wikiloc/map.do?sw=-43.993308489691415%2C172.5656890869141&ne=-43.56496912804993%2C173.40957641601565&sort=last&page=30
Page 30: found 24 cards, added 24 new activities (activity_type: 24, user_name: 24).
Loading page 31: https://www.wikiloc.com/wikiloc/map.do?sw=-43.993308489691415%2C172.5656890869141&ne=-43.56496912804993%2C173.40957641601565&sort=last&page=31
Page 31: found 24 cards, added 24 new activities (activity_type: 24, user_name: 24).
Saved session state: C:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\Data\wikiloc_storage_state.json
Saved 48 extracted activities to C:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\Data\WikiLoc_Downloads\wikiloc_activity_index.csv



In [55]:
if TEST_ACTIVITY_EXPORT:
    try:
        result = subprocess.run(
            [
                sys.executable,
                "wikiloc_activity_data_export.py",
                "--url",
                WIKILOC_ACTIVITY_URL,
                "--out-dir",
                str(OUT_DIR),
                "--profile-dir",
                str(PROFILE_DIR),
                "--storage-state",
                str(STORAGE_STATE_FILE),
                "--index-csv",
                str(WIKILOC_INDEX_CSV),
                "--menu-timeout-ms",
                "5000",
                "--download-timeout-ms",
                "20000",
                "--slow-mo-ms",
                "0",
            ],
            check=True,
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr)
        raise

## Auto Extraction

In [56]:
# # Optional one-time setup for an index file with an Extracted column
# # Example schema: activity_url,Extracted
# index_df = pd.read_csv(WIKILOC_INDEX_CSV)
# index_df["Extracted"] = index_df["Extracted"] = pd.DataFrame({'Extracted': [False] * len(index_df)})
# index_df.to_csv(WIKILOC_INDEX_CSV, index=False)

In [57]:
if AUTO_ACTIVITY_EXPORT:
    index_df = pd.read_csv(WIKILOC_INDEX_CSV)
    if "Extracted" not in index_df.columns:
        index_df["Extracted"] = False
        index_df.to_csv(WIKILOC_INDEX_CSV, index=False)
        index_df = pd.read_csv(WIKILOC_INDEX_CSV)

    MAX_TIMEOUT_RETRIES = 5
    RETRY_WAIT_SECONDS = 60

    for i in range(len(index_df)):
        if bool(index_df.iloc[i].get("Extracted", False)):
            continue

        # if i%10 == 5:
        #     delay_minutes = random.uniform(MIN_DELAY_MINUTES, MAX_DELAY_MINUTES)
        #     print(f"Waiting {delay_minutes:.2f} minutes before next download...")
        #     time.sleep(delay_minutes * 60)

        activity_url = str(index_df.iloc[i]["activity_url"]).strip()
        print(f"Downloading WikiLoc activity {i+1}/{len(index_df)}: {activity_url}")

        attempt = 1
        while attempt <= MAX_TIMEOUT_RETRIES:
            try:
                result = subprocess.run(
                    [
                        sys.executable,
                        "wikiloc_activity_data_export.py",
                        "--url",
                        activity_url,
                        "--out-dir",
                        str(OUT_DIR),
                        "--profile-dir",
                        str(PROFILE_DIR),
                        "--storage-state",
                        str(STORAGE_STATE_FILE),
                        "--index-csv",
                        str(WIKILOC_INDEX_CSV),
                        "--menu-timeout-ms",
                        "5000",
                        "--download-timeout-ms",
                        "60000",
                        "--slow-mo-ms",
                        "0",
                    ],
                    check=True,
                    cwd=str(Path.cwd()),
                    capture_output=True,
                    text=True,
                )
                if result.stdout:
                    print(result.stdout)
                if result.stderr:
                    print(result.stderr)

                # Reload latest CSV first so we keep activity_date/file_name written by downloader.
                index_df = pd.read_csv(WIKILOC_INDEX_CSV)
                if "Extracted" not in index_df.columns:
                    index_df["Extracted"] = False
                index_df.loc[index_df["activity_url"].astype(str).str.strip() == activity_url, "Extracted"] = True
                break
            except subprocess.TimeoutExpired as exc:
                print(f"Timeout while downloading {activity_url} (attempt {attempt}/{MAX_TIMEOUT_RETRIES}).")
                if attempt < MAX_TIMEOUT_RETRIES:
                    print(f"Retrying in {RETRY_WAIT_SECONDS} seconds...")
                    time.sleep(RETRY_WAIT_SECONDS)
                    attempt += 1
                    continue
                raise
            except subprocess.CalledProcessError as exc:
                if exc.stdout:
                    print(exc.stdout)
                if exc.stderr:
                    print(exc.stderr)

                combined_error = f"{exc.stdout or ''}\n{exc.stderr or ''}".lower()
                is_timeout_error = "timeout" in combined_error or "timed out" in combined_error

                if is_timeout_error and attempt < MAX_TIMEOUT_RETRIES:
                    print(
                        f"Timeout-like error for {activity_url} (attempt {attempt}/{MAX_TIMEOUT_RETRIES}). "
                        f"Retrying in {RETRY_WAIT_SECONDS} seconds..."
                    )
                    time.sleep(RETRY_WAIT_SECONDS)
                    attempt += 1
                    continue
                raise

        index_df.to_csv(WIKILOC_INDEX_CSV, index=False)